# Complete Pipeline: AI Human Activity Recognition

This notebook demonstrates the **entire workflow** of the project from start to finish, including dataset preparation, preprocessing, training, evaluation, and inference. 

> **Note:** Many of these steps (like downloading the dataset and training the model) can take a significant amount of time and disk space. The bash commands are commented out by default; uncomment them to execute the full pipeline step by step.

## 1. Setup & Configuration
First, let's load our configuration to see the paths and hyperparameters used throughout the project.

In [ ]:
import os
import sys

# Add project root to python path
project_root = os.path.abspath('.')
if project_root not in sys.path:
    sys.path.append(project_root)

import config
print("Loaded configuration!")
print(f"Activity Classes: {config.ACTIVITY_CLASSES}")
print(f"Sequence Length: {config.SEQUENCE_LENGTH} frames")
print(f"Model Architecture: {config.MODEL_CONFIG.architecture}")

## 2. Dataset Preparation & Preprocessing

The preprocessing pipeline consists of several sequential steps. You can run these commands via `!` directly in this notebook.

1. **Download Dataset:** Downloads the UCF101 dataset (several GBs).
2. **Extract Frames:** Converts videos into a series of images (frames).
3. **Generate Landmarks:** Uses MediaPipe to extract 33 skeletal landmarks from each frame.
4. **Create Sequences:** Groups consecutive frames into fixed-length sequences (e.g., 30 frames) for the LSTM model.
5. **Split Dataset:** Divides the data into training, validation, and testing sets.

In [ ]:
# Step 2.1: Download Dataset
# !python -m preprocessing.download_dataset --dataset ucf101

In [ ]:
# Step 2.2: Extract Frames
# !python -m preprocessing.extract_frames --input data/raw/ucf101 --frames-per-video 60

In [ ]:
# Step 2.3: Generate Pose Landmarks
# !python -m preprocessing.generate_landmarks

In [ ]:
# Step 2.4: Build fixed-length sequences (with data augmentation)
# !python -m preprocessing.create_sequences

In [ ]:
# Step 2.5: Split into Train/Val/Test
# !python -m preprocessing.split_dataset

## 3. Training the Model

Once the data is preprocessed into `.npy` sequence files, we can train the neural network model. 
The training script includes early stopping, learning rate scheduling, and checkpointing automatically.

In [ ]:
# Run the training script
# !python -m training.train --epochs 50 --batch-size 32 --lr 1e-3

# To resume training from a checkpoint, you can use:
# !python -m training.train --resume saved_models/best_model.keras --epochs 50

## 4. Evaluation

After training, evaluate the model on the test dataset to generate a confusion matrix, ROC curves, and a classification report. These outputs will be saved in the `outputs/` folder.

In [ ]:
# Evaluate the model
# !python -m training.evaluate

## 5. Programmatic Inference

Here we show how you can load the trained model within python to make predictions on new data, such as a video file or a webcam feed.

In [ ]:
from models.inference import ActivityPredictor

try:
    predictor = ActivityPredictor()
    print("Activity Predictor loaded successfully!")
except Exception as e:
    print("Predictor could not be loaded. Please ensure you have trained a model first.")
    print(f"Error: {e}")

In [ ]:
# Example: Inference on a Video File
video_path = "path/to/your/test/video.mp4"

if os.path.exists(video_path) and 'predictor' in locals():
    print(f"Running inference on {video_path}...")
    result = predictor.predict_video(video_path)
    
    print("\n--- Overall Prediction ---")
    print(f"Predicted Activity: {result['activity']}")
    print(f"Confidence: {result['confidence']:.2f}")
else:
    print(f"Please provide a valid video path for testing.")

## 6. Streamlit Web App Deployment

The project includes a fully functional Streamlit application that provides a UI for both video upload and live webcam inference.

In [ ]:
# To launch the Streamlit app, run this command in your terminal:
# !streamlit run app.py
print("Run `streamlit run app.py` in your terminal to start the web app.")